# 📊 Customer Churn Analysis
**Telco Customer Dataset — Exploratory Data Analysis & Business Insights**

---
**Objective:** Identify patterns and factors contributing to customer churn and provide actionable retention strategies.

**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

os.makedirs('../visuals', exist_ok=True)

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Data Cleaning

In [ ]:
# --- Missing Values ---
print('Missing values before fix:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# TotalCharges contains whitespace strings — convert and fill
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True)

# --- Duplicates ---
print(f'\nDuplicate rows: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)

# --- Fix Data Types ---
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('\nData Cleaning Complete ✅')
print(f'Final shape: {df.shape}')
df.dtypes

## 3. Key Metrics & Customer Segmentation

In [ ]:
churn_rate = df['Churn'].mean() * 100
avg_monthly = df['MonthlyCharges'].mean()
avg_tenure = df['tenure'].mean()

print(f'📌 Churn Rate          : {churn_rate:.2f}%')
print(f'💰 Avg Monthly Charges : ${avg_monthly:.2f}')
print(f'📅 Avg Tenure (months) : {avg_tenure:.1f}')

# Customer Segmentation
def segment(row):
    if row['tenure'] <= 12:   return 'New (<1yr)'
    elif row['tenure'] <= 36: return 'Mid-Term (1-3yr)'
    else:                     return 'Long-Term (3yr+)'

df['Segment'] = df.apply(segment, axis=1)
seg = df.groupby('Segment')['Churn'].agg(['mean','count']).reset_index()
seg.columns = ['Segment', 'Churn Rate', 'Count']
seg['Churn Rate'] = (seg['Churn Rate'] * 100).round(2)
print('\nCustomer Segments:')
seg

In [ ]:
print('Statistical Summary:')
display(df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2))

print('\nMean values — Churned vs Retained:')
comp = df.groupby('Churn')[['MonthlyCharges','tenure','TotalCharges']].mean().round(2)
comp.index = ['Retained', 'Churned']
display(comp)

## 4. Visualizations

In [ ]:
# 4.1 Churn Distribution
COLORS = ['#2ecc71', '#e74c3c']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Churn Distribution', fontsize=14, fontweight='bold')

counts = df['Churn'].value_counts()
axes[0].bar(['Retained','Churned'], counts.values, color=COLORS, width=0.5, edgecolor='white')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+50, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Retained','Churned'], colors=COLORS,
            autopct='%1.1f%%', startangle=140, wedgeprops={'edgecolor':'white'})
axes[1].set_title(f'Overall Churn Rate: {churn_rate:.1f}%')

plt.tight_layout()
plt.savefig('../visuals/01_churn_distribution.png')
plt.show()

In [ ]:
# 4.2 Gender vs Churn
fig, ax = plt.subplots(figsize=(7, 5))
g = df.groupby('gender')['Churn'].mean() * 100
bars = ax.bar(g.index, g.values, color=['#3498db','#e91e63'], width=0.4, edgecolor='white')
ax.set_title('Churn Rate by Gender', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 35)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/02_gender_vs_churn.png')
plt.show()

In [ ]:
# 4.3 Contract Type vs Churn
fig, ax = plt.subplots(figsize=(9, 5))
c = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False) * 100
bars = ax.bar(c.index, c.values, color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='white', width=0.5)
ax.set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 55)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/03_contract_vs_churn.png')
plt.show()

In [ ]:
# 4.4 Monthly Charges vs Churn
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Monthly Charges vs Churn', fontsize=14, fontweight='bold')

retained = df[df['Churn']==0]['MonthlyCharges']
churned  = df[df['Churn']==1]['MonthlyCharges']

axes[0].hist(retained, bins=30, alpha=0.6, color='#2ecc71', label='Retained')
axes[0].hist(churned,  bins=30, alpha=0.6, color='#e74c3c', label='Churned')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Distribution')

axes[1].boxplot([retained, churned], labels=['Retained','Churned'],
                patch_artist=True, boxprops=dict(facecolor='#f0f0f0'),
                medianprops=dict(color='black', linewidth=2))
axes[1].set_ylabel('Monthly Charges ($)')
axes[1].set_title('Boxplot')

plt.tight_layout()
plt.savefig('../visuals/04_monthly_charges_vs_churn.png')
plt.show()

In [ ]:
# 4.5 Tenure vs Churn
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Tenure vs Churn', fontsize=14, fontweight='bold')

axes[0].hist(df[df['Churn']==0]['tenure'], bins=30, alpha=0.6, color='#2ecc71', label='Retained')
axes[0].hist(df[df['Churn']==1]['tenure'], bins=30, alpha=0.6, color='#e74c3c', label='Churned')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Distribution')

seg_order = ['New (<1yr)', 'Mid-Term (1-3yr)', 'Long-Term (3yr+)']
seg_rates = df.groupby('Segment')['Churn'].mean().reindex(seg_order) * 100
bars = axes[1].bar(seg_order, seg_rates.values,
                   color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='white', width=0.5)
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Customer Segment')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{bar.get_height():.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../visuals/05_tenure_vs_churn.png')
plt.show()

In [ ]:
# 4.6 Payment Method vs Churn
fig, ax = plt.subplots(figsize=(11, 5))
p = df.groupby('PaymentMethod')['Churn'].mean().sort_values() * 100
bars = ax.barh(p.index, p.values, color=['#2ecc71','#27ae60','#f39c12','#e74c3c'], edgecolor='white')
ax.set_title('Churn Rate by Payment Method', fontsize=13, fontweight='bold')
ax.set_xlabel('Churn Rate (%)')
for bar in bars:
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/06_payment_method_vs_churn.png')
plt.show()

In [ ]:
# 4.7 Correlation Heatmap
fig, ax = plt.subplots(figsize=(11, 8))
corr_df = df[['tenure','MonthlyCharges','TotalCharges','Churn']].copy()
corr_df['IsFiberOptic']    = (df['InternetService'] == 'Fiber optic').astype(int)
corr_df['IsMonthToMonth']  = (df['Contract'] == 'Month-to-month').astype(int)
corr_df['IsSenior']        = (df['SeniorCitizen'] == 'Yes').astype(int)
corr_df['ElectronicCheck'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
corr_df['PaperlessBilling']= (df['PaperlessBilling'] == 'Yes').astype(int)

mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/07_correlation_heatmap.png')
plt.show()

In [ ]:
# 4.8 Internet Service vs Churn
fig, ax = plt.subplots(figsize=(9, 5))
inet = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False) * 100
bars = ax.bar(inet.index, inet.values, color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='white', width=0.5)
ax.set_title('Churn Rate by Internet Service', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 50)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/08_internet_service_vs_churn.png')
plt.show()

## 5. Key Factors Influencing Churn

In [ ]:
factors = {
    'Month-to-Month Contract'   : df[df['Contract']=='Month-to-month']['Churn'].mean(),
    'Electronic Check Payment'  : df[df['PaymentMethod']=='Electronic check']['Churn'].mean(),
    'Fiber Optic Internet'      : df[df['InternetService']=='Fiber optic']['Churn'].mean(),
    'Paperless Billing'         : df[df['PaperlessBilling']=='Yes']['Churn'].mean(),
    'Senior Citizen'            : df[df['SeniorCitizen']=='Yes']['Churn'].mean(),
    'No Online Security'        : df[df['OnlineSecurity']=='No']['Churn'].mean(),
    'No Tech Support'           : df[df['TechSupport']=='No']['Churn'].mean(),
}
factors_df = pd.DataFrame.from_dict(factors, orient='index', columns=['Churn Rate'])
factors_df['Churn Rate'] = (factors_df['Churn Rate'] * 100).round(2)
factors_df = factors_df.sort_values('Churn Rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(factors_df.index, factors_df['Churn Rate'],
               color='#e74c3c', edgecolor='white')
ax.axvline(churn_rate, color='gray', linestyle='--', label=f'Avg Churn {churn_rate:.1f}%')
ax.set_title('Churn Rate by Risk Factor', fontsize=13, fontweight='bold')
ax.set_xlabel('Churn Rate (%)')
ax.legend()
for bar in bars:
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/09_risk_factors.png')
plt.show()
display(factors_df)

## 6. Business Insights & Recommendations

### 🔍 Key Findings

| # | Finding | Churn Rate |
|---|---------|------------|
| 1 | Month-to-month contract customers | ~43% |
| 2 | Electronic check payment users | ~45% |
| 3 | Fiber optic internet customers | ~42% |
| 4 | New customers (≤12 months tenure) | ~48% |
| 5 | No online security or tech support | ~42% |
| 6 | Senior citizens | ~41% |
| 7 | Paperless billing customers | ~34% |

### 💡 Recommendations

1. **Contract Upgrade Campaign** — Offer discounts/loyalty rewards to move month-to-month customers to annual/2-year plans.
2. **Auto-Pay Incentive** — Provide a small monthly discount for switching from electronic check to automatic bank/card payment.
3. **Fiber Optic Service Quality Review** — Audit fiber optic pricing, speeds, and support. Consider competitive pricing analysis.
4. **Early Onboarding Program** — Dedicated onboarding specialists for first 90 days; proactive check-in calls.
5. **Security & Support Bundles** — Bundle tech support + online security at reduced cost to increase stickiness.
6. **Senior Citizen Retention Plan** — Dedicated support line and simplified billing for seniors.

### ✅ Conclusion

The overall churn rate is **~26.5%**, which is significantly high for a subscription business. The analysis reveals that churn is strongly driven by **short-term contracts**, **payment friction** (electronic check), and **early tenure**. Targeted interventions in contract upselling, onboarding quality, and service bundling could realistically reduce churn by **8–12 percentage points**, translating to significant revenue retention.